In [ ]:
from pathlib import Path

import pandas as pd
import py3Dmol
from IPython.display import Image
from ipywidgets import fixed, interact, interactive
from pymol import cmd
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdDepictor, rdDistGeom
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.PandasTools import ChangeMoleculeRendering, FrameToGridImage
from rdkit.Chem.rdmolfiles import MolFromMolBlock, MolFromSmiles
from rdkit.Chem.rdmolops import AddHs, RemoveHs

In [ ]:
folder = Path("/homes/buttensc/Projects/semla-flow/predictions/")
file_table = folder / "unconditional/geoldm/geoldm_100000_predictions.csv"
file_mols = folder / "unconditional/geoldm/geoldm_100000_predictions.sdf"

In [ ]:
with open(file_mols, "r") as file:
    mol_blocks = file.read().rstrip().rstrip("\n").rstrip("\n").rstrip("$$$$").split("$$$$\n")
print(len(mol_blocks))

In [ ]:
df = pd.read_csv(file_table, low_memory=False)
df["method"] = "geoldm"
df["idx"] = df["name"].str.split("_").str[-1].astype(int)
print(len(df))

In [ ]:
df = df[df.chemical.fillna(False)].copy()

df["mol_flat"] = df.idx.apply(lambda i: MolFromMolBlock(mol_blocks[i], sanitize=True))
df.mol_flat.apply(lambda m: m.RemoveAllConformers())
print(len(df))

In [ ]:
FrameToGridImage(df.sample(30), column="mol_flat", subImgSize=(400, 400), molsPerRow=5)

In [ ]:
import py3Dmol
from ipywidgets import fixed, interact, interactive
from rdkit import Chem
from rdkit.Chem import AllChem


def drawit(molblock, viewer, confId=-1):
    viewer.removeAllModels()
    viewer.addModel(molblock, "sdf")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    return viewer.show()


viewer = py3Dmol.view(width=400, height=400)
viewer.setBackgroundColor("0xffffff")


@interact
def show_molecules(i=(0, len(df) - 1)):
    index = df.loc[i, "idx"]
    drawit(mol_blocks[index], viewer)
